# Proper Evaluation

## 1. Setup

In [1]:
import json
import sys
import os
import time
import pandas as pd
from pathlib import Path
from typing import List, Dict
from dotenv import load_dotenv
from rouge_score import rouge_scorer

## 2. Configuration & API Setup

In [2]:

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
os.chdir(project_root)
sys.path.insert(0, str(project_root))

from src.services.llm_services import load_config, get_llm, validate_api_keys
from src.utils.cost_tracker import total_cost

load_dotenv()
config = load_config("src/config/config.yaml")
validate_api_keys(config)

print(f"📂 Project Root: {project_root}")
judge_llm = get_llm(config)

e:\my career life\AI Enginert Essential\Mini project 01\src\services\llm_services.py:375: UserWarning: ⚠️  COHERE_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")


📂 Project Root: e:\my career life\AI Enginert Essential\Mini project 01


## 3. Load Evaluation Results

In [3]:
rag_path = config.get('eval_results_path', 'artifacts/test/rag_evaluation_results.json')
finetune_path = config.get('finetune_results_path', 'artifacts/test/finetune_evaluation_results.json')

def load_json(path):
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    print(f"⚠️ Warning: {path} not found.")
    return []

rag_results = load_json(rag_path)
finetune_results = load_json(finetune_path)

print(f"✅ Loaded {len(rag_results)} RAG cases")
print(f"✅ Loaded {len(finetune_results)} Finetune cases")

✅ Loaded 10 RAG cases
✅ Loaded 10 Finetune cases


## 4. Metrics Implementation

In [4]:
rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def get_rouge_l(target, prediction):
    scores = rouge_scorer_obj.score(target, prediction)
    return scores['rougeL'].fmeasure

def evaluate_with_llm_judge(question, target, prediction):
    prompt = f"""You are an expert evaluator. Evaluate the following AI response based on the provided reference answer.

Question: {question}
Reference Answer: {target}
AI Prediction: {prediction}

Provide your evaluation in the following format:
SCORE: [1-5]
REASONING: [Brief explanation]"""
    
    try:
        response = judge_llm.invoke(prompt)
        res_text = response.content if hasattr(response, 'content') else str(response)
        
        # Extract score
        score = 0
        for line in res_text.split('\n'):
            if "SCORE:" in line:
                digits = ''.join(filter(str.isdigit, line))
                if digits: score = int(digits)
        
        return score, res_text
    except Exception as e:
        print(f"⚠️ Judge API Error: {e}")
        return None, str(e)

## 5. Run Evaluation

In [5]:

def run_eval_on_dataset(dataset, name, limit=10):
    print(f"🚀 Evaluating {name} ({min(len(dataset), limit)} cases)...")
    results = []
    for i, item in enumerate(dataset[:limit]):
        q = item.get('question', '')
        target = item.get('actual_answer', '')
        pred = item.get('generated_answer', '')
        
        rouge_l = get_rouge_l(target, pred)
        score, reason = evaluate_with_llm_judge(q, target, pred)
        
        results.append({
            "question": q,
            "rouge_l": rouge_l,
            "judge_score": score,
            "reasoning": reason
        })
        print(f"  [{i+1}/{limit}] Score: {score} | ROUGE: {rouge_l:.4f}")
        time.sleep(0.5) # Avoid rate limits
    return results

print("Comparing RAG vs Intern (Finetuned)...\n")
rag_eval = run_eval_on_dataset(rag_results, "RAG (The Librarian)")
print("\n")
finetune_eval = run_eval_on_dataset(finetune_results, "Intern (Finetuned)")

Comparing RAG vs Intern (Finetuned)...

🚀 Evaluating RAG (The Librarian) (10 cases)...


  [1/10] Score: 5 | ROUGE: 0.1429
  [2/10] Score: 5 | ROUGE: 0.4800
  [3/10] Score: 1 | ROUGE: 0.0000
  [4/10] Score: 5 | ROUGE: 0.4571
  [5/10] Score: 4 | ROUGE: 0.7368
  [6/10] Score: 5 | ROUGE: 0.0299
  [7/10] Score: 5 | ROUGE: 0.2857
  [8/10] Score: 5 | ROUGE: 0.1386
  [9/10] Score: 5 | ROUGE: 0.3750
  [10/10] Score: 5 | ROUGE: 0.5000


🚀 Evaluating Intern (Finetuned) (10 cases)...
  [1/10] Score: 5 | ROUGE: 0.3333
  [2/10] Score: 2 | ROUGE: 0.2857
  [3/10] Score: 1 | ROUGE: 0.0000
  [4/10] Score: 1 | ROUGE: 0.3125
  [5/10] Score: 2 | ROUGE: 0.4364
  [6/10] Score: 4 | ROUGE: 0.0000
  [7/10] Score: 1 | ROUGE: 0.4000
  [8/10] Score: 5 | ROUGE: 0.1481
  [9/10] Score: 5 | ROUGE: 0.1818
  [10/10] Score: 1 | ROUGE: 0.0000


## 6. Results Analysis

In [ ]:

def get_metrics(eval_data):
    avg_rouge = sum(d['rouge_l'] for d in eval_data) / len(eval_data)
    avg_score = sum(d['judge_score'] for d in eval_data if d['judge_score']) / len([d for d in eval_data if d['judge_score']])
    return avg_rouge, avg_score

rag_rouge, rag_score = get_metrics(rag_eval)
ft_rouge, ft_score = get_metrics(finetune_eval)

summary = pd.DataFrame({
    "Model": ["The Librarian (RAG)", "The Intern (Finetuned)"],
    "ROUGE-L": [rag_rouge, ft_rouge],
    "LLM-as-a-Judge": [rag_score, ft_score]
})

print("\n=== FINAL SHOWDOWN ===")
print(summary.to_string(index=False))

# Save evaluation full report
report_path = "artifacts/report/final_arena_report.json"
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump({"rag": rag_eval, "finetune": finetune_eval}, f, indent=4)
print(f"\nFull report saved to {report_path}")


=== FINAL SHOWDOWN ===
                 Model  ROUGE-L  LLM-as-a-Judge
   The Librarian (RAG) 0.314602             4.5
The Intern (Finetuned) 0.209788             2.7

Full report saved to artifacts/report/final_arena_report.json


## 7. Cost Analysis calculation and markdown summary. 

Note on Performance Metrics: "The cost and latency measurement logic was implemented independently within the RAG and Finetuning notebooks. For the RAG pipeline, costs were calculated dynamically based on real-time token consumption. However, for the Finetuned model, the cost reflects the total training overhead incurred on Google Colab (T4 GPU). Since these training costs are external to the inference process, they were manually extracted and integrated into this report for a comprehensive financial comparison."

In [9]:
# Constants based on previous calculations
finetuned_computational_cost = 0.73  # Total T4 Training cost on Colab
rag_inference_cost = total_cost()    # Dynamic token cost from RAG evaluation

# Final calculation
total_project_cost = finetuned_computational_cost + rag_inference_cost

print(f"--- Financial Breakdown ---")
print(f"Finetuning (Training): ${finetuned_computational_cost:.2f}")
print(f"RAG (Inference):      ${rag_inference_cost:.5f}")
print(f"---------------------------")
print(f"Total Project Cost:    ${total_project_cost:.5f}")

# Save evaluation full report
report_path = "artifacts/report/final_cost_report.json"
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump({"finetuned_computational_cost":finetuned_computational_cost,"rag_inference_cost":rag_inference_cost,"total_project_cost":total_project_cost}, f, indent=4)
print(f"\nFull report saved to {report_path}")

Successfully loaded JSON data from e:\my career life\AI Enginert Essential\Mini project 01\./artifacts/test/rag_evaluation_results.json
--- Financial Breakdown ---
Finetuning (Training): $0.73
RAG (Inference):      $0.00400
---------------------------
Total Project Cost:    $0.73400

Full report saved to artifacts/report/final_cost_report.json
